In [0]:
%sql
USE CATALOG medalhao;

/*
  Criando o Banco de dados da camada bronze
*/
CREATE SCHEMA IF NOT EXISTS gold;

USE SCHEMA gold;

In [0]:
from pyspark.sql.functions import col, count, count_distinct, sum, round, year, month, coalesce, lit

# 1. LEITURA DOS DADOS DA CAMADA SILVER
df_pedido_total = spark.read.table("silver.fat_pedido_total")
df_itens = spark.read.table("silver.fat_itens_pedidos")
df_produtos = spark.read.table("silver.dim_produtos")
df_traducao = spark.read.table("silver.dim_categoria_produtos_traducao")

# 2. ENRIQUECIMENTO TEMPORAL
# Justificativa: Melhorando o agrupamento para análise de séries temporais.
df_pedido_total = (
    df_pedido_total
    .withColumn("ano_venda", year("data_pedido"))
    .withColumn("mes_venda", month("data_pedido"))
)

# 3. CONSOLIDAÇÃO DE CATEGORIAS
# Justificativa: Agregando a categoria ao item para permitir a visão por setor de produto.
df_itens_com_categoria = (
    df_itens
    .join(df_produtos, "id_produto", "left")
    .join(
        df_traducao,
        df_produtos.categoria_produto == df_traducao.nome_produto_pt,
        "left"
    )
    .select(
        col("id_pedido"),
        coalesce(col("nome_produto_pt"), lit("Sem Categoria")).alias("categoria_produto")
    )
)

# 4. CRIAÇÃO DA BASE INTEGRADA
df_base = df_pedido_total.join(df_itens_com_categoria, "id_pedido", "inner")

# AGREGAÇÃO DE ITENS (Granularidade: Pedido x Item)
df_itens_agg = (
    df_base
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(count("*").alias("qtd_itens_vendidos"))
)

# AGREGAÇÃO DE PEDIDOS E RECEITA (Granularidade: Pedido Único)
# Justificativa: Evitar a duplicação da receita quando um pedido possui múltiplos itens.
df_pedidos_agg = (
    df_base
    .dropDuplicates(["id_pedido"]) # Elimina duplicidades
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        count_distinct("id_pedido").alias("total_pedidos"),
        sum(coalesce(col("valor_total_pago_brl"), lit(0))).alias("receita_total_brl"),
        sum(coalesce(col("valor_total_pago_usd"), lit(0))).alias("receita_total_usd")
    )
)

# 5. CÁLCULO DE MÉTRICAS
df_gold_comercial = (
    df_pedidos_agg
    .join(df_itens_agg, ["ano_venda", "mes_venda", "categoria_produto"])
    .withColumn("receita_total_brl", round(col("receita_total_brl"), 2))
    .withColumn("receita_total_usd", round(col("receita_total_usd"), 2))
    # Regra de Negócio: Ticket Médio por categoria
    .withColumn(
        "ticket_medio_brl",
        round(col("receita_total_brl") / col("total_pedidos"), 2)
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

# 6. ESCREVENDO NA CAMADA GOLD
df_gold_comercial.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fat_vendas_comercial")

In [0]:
from pyspark.sql.functions import col, count, coalesce, lit

# 1. PROCESSAMENTO DO RANKING
df_rank_produtos = (
    # Inicia pela tabela de fatos de itens (granularidade de venda)
    df_itens
    # Busca o nome do produto na dimensão de produtos
    .join(df_produtos, "id_produto", "left")
    # Busca a tradução da categoria para exibição amigável
    .join(
        df_traducao,
        df_produtos.categoria_produto == df_traducao.nome_produto_pt,
        "left"
    )
    # 2. AGRUPAMENTO
    # Justificativa: Agrupamos por nome e categoria para consolidar o volume total por item.
    .groupBy(
        col("nome_produto"),
        coalesce(col("nome_produto_pt"), lit("Sem Categoria")).alias("categoria_produto")
    )
    # 3. AGREGAÇÃO
    .agg(
        count("*").alias("quantidade_vendida")
    )
)

# 4. FILTRAGEM DOS TOP 5
# Justificativa: Ordenação descendente para trazer os maiores volumes para o topo.
df_top5_mais = df_rank_produtos.orderBy(col("quantidade_vendida").desc()).limit(5)

# 5. VISUALIZAÇÃO
# O comando display é utilizado no Databricks para renderizar tabelas e gráficos rapidamente.
display(df_top5_mais)

nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


In [0]:
df_top5_menos = (
    df_rank_produtos
    .filter(col("quantidade_vendida") > 0)
    .orderBy(col("quantidade_vendida").asc())
    .limit(5)
)

display(df_top5_menos)

nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


In [0]:
from pyspark.sql.functions import col, count, avg, sum, when, round, coalesce, lit

# 1. LEITURA DAS TABELAS (CAMADA SILVER)
df_avaliacoes = spark.read.table("silver.fat_avaliacoes_pedidos")
df_itens = spark.read.table("silver.fat_itens_pedidos")
df_produtos = spark.read.table("silver.dim_produtos")
df_vendedores = spark.read.table("silver.dim_vendedores")
df_traducao = spark.read.table("silver.dim_categoria_produtos_traducao")

# 2. CONSTRUÇÃO DA BASE INTEGRADA
# Justificativa: Cruzamento necessário para saber QUEM vendeu e O QUE foi avaliado.
df_base_avaliacoes = (
    df_avaliacoes
    .join(df_itens, "id_pedido", "left")
    .join(df_produtos, "id_produto", "left")
    .join(df_vendedores, "id_vendedor", "left")
    .join(
        df_traducao,
        df_produtos.categoria_produto == df_traducao.nome_produto_pt,
        "left"
    )
)

# 3. AGREGAÇÃO E CÁLCULO DE MÉTRICAS (CAMADA GOLD)
df_gold_avaliacoes = (
    df_base_avaliacoes
    .groupBy(
        coalesce(col("nome_produto_pt"), lit("Sem Categoria")).alias("categoria_produto"),
        col("nome_vendedor"),
        col("estado").alias("estado_vendedor")
    )
    .agg(
        count("*").alias("total_avaliacoes"),
        # Regra: Média de satisfação arredondada
        round(avg("nota_avaliacao").cast("double"), 2).alias("avaliacao_media"),
        # Regra de Negócio: Notas 4 e 5 são consideradas promotoras/positivas
        sum(when(col("nota_avaliacao").cast("double") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        # Regra de Negócio: Notas 1 e 2 são consideradas detratoras/negativas
        sum(when(col("nota_avaliacao").cast("double") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    # 4. CÁLCULO DE PERCENTUAL (SCORE)
    .withColumn(
    "percentual_satisfacao",
        when(col("total_avaliacoes") > 0,
            round(col("total_avaliacoes_positivas") / col("total_avaliacoes") * 100, 2)
        ).otherwise(0)
    )
)

# 5. PERSISTÊNCIA E EXIBIÇÃO
# Salva como tabela Delta para consumo em dashboards de CX (Customer Experience)
df_gold_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("gold.fat_avaliacoes_clientes")
